<a href="https://colab.research.google.com/github/testimonyadewale/msc-grocery-forecasting/blob/main/MSc_models_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded')

# Load the processed dataset — no need to redo feature engineering
df = pd.read_csv('/content/drive/MyDrive/MSc_Data/train_processed.csv')
df['date'] = pd.to_datetime(df['date'])

print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All libraries loaded
Loaded: (730500, 20)
Columns: ['date', 'store', 'item', 'sales', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_365', 'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_84', 'rolling_std_7', 'rolling_std_28', 'day_of_week', 'week_of_year', 'month', 'year', 'quarter', 'is_weekend']


In [27]:
feature_cols = [
    'store', 'item',
    'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_365',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_mean_84',
    'rolling_std_7', 'rolling_std_28',
    'day_of_week', 'week_of_year', 'month',
    'year', 'quarter', 'is_weekend'
]

X = df[feature_cols]
y = df['sales']

train_mask = df['date'] < '2017-01-01'
test_mask  = df['date'] >= '2017-01-01'

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Training rows: {len(X_train)}')
print(f'Test rows:     {len(X_test)}')

Training rows: 548000
Test rows:     182500


In [28]:
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name}:')
    print(f'  MAE:  {mae:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  R2:   {r2:.4f}')
    return {'Model': name, 'MAE': round(mae,4),
            'RMSE': round(rmse,4), 'R2': round(r2,4)}

print('evaluate function ready')

evaluate function ready


In [29]:
ma_pred = X_test['rolling_mean_28'].fillna(y_train.mean())
results_ma = evaluate('Moving Average (Baseline)', y_test, ma_pred)

Moving Average (Baseline): MAE=9.2373  RMSE=12.2565  R2=0.8491
